# Create excel sheet for GENEVA dataset

In [2]:
import pandas as pd
import os

# Function to load data and create the new dataframe
def create_combined_dataframe(file_path):
    # Load the specific sheets

    file_1 = pd.read_excel(os.path.join(file_path, 'SYMONE_original_farnaz.xlsx'))
    print(file_1.columns)
    file_2 = pd.read_excel(os.path.join(file_path, 'TRACON_original_farnaz.xlsx'))

    # --- FILTER OUT UNWANTED ROWS IN file 1: Berlin cohort and 3rd session ---
    # 1) drop t3 in "redcap_event_name"
    mask_no_t3 = ~file_1['"redcap_event_name"'].astype(str).str.contains('t3', case=False, na=False)
    # 2) drop BS in "record_id"
    mask_no_BS = ~file_1['"record_id"'].astype(str).str.contains('BS', case=False, na=False)
    mask_no_BC = ~file_1['"record_id"'].astype(str).str.contains('BC', case=False, na=False)

    file_1 = file_1[mask_no_t3 & mask_no_BS & mask_no_BC].copy()

    # Concatenate the columns from both sheets using the column names
    sub_id_1 = file_1['"record_id"']  # Column for sub_id in sheet 1
    sub_id_2 = file_2['Subject']  # Column for sub_id in sheet 2
    print(sub_id_1.head())

    # Determine session based on 'redcap_event_name' column
    session_1 = file_1['"redcap_event_name"'].apply(lambda x: 1 if 't1' in str(x) else (2 if 't2' in str(x) else None))
    session_2 = pd.Series(1, index=file_2.index)  # All entries in sheet 2 correspond to session 1
    # Extract the other columns using the provided names

    # For file_1
    file_1['"ris_eqtot"'] = (
        file_1['"ris_eqtot"']
        .astype(str)
        .str.replace(',', '.', regex=False)   # turn 3,333 → 3.333
    )
    file_1['"ris_eqtot"'] = pd.to_numeric(file_1['"ris_eqtot"'], errors="coerce")

    # For file_2
    file_2['Risperidone equivalents'] = (
        file_2['Risperidone equivalents']
        .astype(str)
        .str.replace(',', '.', regex=False)
    )
    file_2['Risperidone equivalents'] = pd.to_numeric(file_2['Risperidone equivalents'], errors="coerce")


    # NEED Risperidone score from sheet 1
    # As numeric Series
    risperidon_score_1 = pd.to_numeric(file_1['"ris_eqtot"'], errors="coerce")
    risperidon_score_2 = pd.to_numeric(file_2['Risperidone equivalents'], errors="coerce")


    bacs_total_1 = file_1['"bacs_total_t"']  # Column for bacs_total
    bacs_total_2 = file_2['bacs_compo_t_score']  # Column for bacs_total

    bacs_att_1 = file_1['"bacs_att"']  # Column for symbol coding task
    bacs_att_2 = file_2['bacs_att']  # Column for symbol coding task

    panss_positive_1 = file_1['"panss_pos_total"']  # Column for panss_positive
    panss_positive_2 = file_2['panss_positive_score']  # Column for panss_positive
    panss_negative_1 = file_1['"panss_neg_total"']  # Column for panss_negative
    cols_epa = ['"panss_negativ_1"', '"panss_negativ_2"', '"panss_negativ_3"', '"panss_negativ_4"', '"panss_negativ_6"']
    panss_negative_EPA_1 = file_1[cols_epa].sum(axis=1)

    cols_indrit = ['"panss_negativ_2"', '"panss_negativ_4"']
    panss_apathy_Indrit_1 = file_1[cols_indrit].sum(axis=1)

    panss_negative_2 = file_2['panss_negative_score']  # Column for panss_negative
    cols_epa = ['panss_negativ_1', 'panss_negativ_2', 'panss_negativ_3', 'panss_negativ_4', 'panss_negativ_6']
    cols_indrit = ['panss_negativ_2', 'panss_negativ_4']

    panss_negative_EPA_2 = file_2[cols_epa].sum(axis=1)
    panss_apathy_Indrit_2 = file_2[cols_indrit].sum(axis=1)


    age_years_1 = file_1['"db_age"']  # Column for age_years
    age_years_2 = file_2['Age (yo)']  # Column for age_years


    sex_1 = file_1['"db_sex"'].apply(lambda x: "m" if '1' in str(x) else ("f" if '0' in str(x) else None))  # Column for sex
    sex_2 = file_2['Gender (M=1)'].apply(lambda x: "m" if '1' in str(x) else ("f" if '0' in str(x) else None))  # Column for sex

    bnss_apathy_1 = file_1['"bnss_apathy"']  # Column for bnss_apathy
    bnss_apathy_2 = file_2['bnss_apathie']  # Column for bnss_apathy

    # Create the 'group' column based on the value of the 'DX' column
    group_1 = file_1['"record_id"'].apply(lambda x: "Schizophrenia" if 'GS' in str(x) else ("Healthy Control" if 'GC' in str(x) else None))
    group_2 = file_2['Group'].apply(lambda x: "Schizophrenia" if 'patient' in str(x) else ("Healthy Control" if 'control' in str(x) else None))
    # Combine all columns into a new dataframe
    combined_df = pd.DataFrame({
        'sub_id': (
            pd.concat([sub_id_1, sub_id_2], axis=0)
            .astype(str)
            .str.replace('"', '', regex=False)
            .str.replace(r'\.0$', '', regex=True)  # remove trailing ".0"
            .reset_index(drop=True)
        ),
        'session': pd.concat([session_1, session_2], axis=0).reset_index(drop=True),
        'risperidon_score': pd.concat([risperidon_score_1, risperidon_score_2], axis=0).reset_index(drop=True),
        'bacs_total': pd.concat([bacs_total_1, bacs_total_2], axis=0).reset_index(drop=True),
        'bacs_att': pd.concat([bacs_att_1, bacs_att_2], axis=0).reset_index(drop=True),
        'panss_positive': pd.concat([panss_positive_1, panss_positive_2], axis=0).reset_index(drop=True),
        'panss_negative': pd.concat([panss_negative_1, panss_negative_2], axis=0).reset_index(drop=True),
        'panss_negative_EPA': pd.concat([panss_negative_EPA_1, panss_negative_EPA_2], axis=0).reset_index(drop=True),
        'panss_apathy_Indrit': pd.concat([panss_apathy_Indrit_1, panss_apathy_Indrit_2], axis=0).reset_index(drop=True),
        'age_years': pd.concat([age_years_1, age_years_2], axis=0).reset_index(drop=True),
        'sex': pd.concat([sex_1, sex_2], axis=0).reset_index(drop=True),
        'bnss_apathy': pd.concat([bnss_apathy_1, bnss_apathy_2], axis=0).reset_index(drop=True),
        'group': pd.concat([group_1, group_2], axis=0).reset_index(drop=True)
    })

    return combined_df

# Define the file path
file_path = '/Volumes/M/Geneva_Schizophrenia_Begue/metadata'

# Call the function to create the combined dataframe
combined_df = create_combined_dataframe(file_path)
combined_df.to_excel('/Volumes/M/Geneva_Schizophrenia_Begue/metadata/metadata_GENEVA_dataset.xlsx', index=False)

# Display the resulting dataframe (you can also save it to a file or do further analysis)
print(combined_df.head())






Index(['""', '"record_id"', '"redcap_event_name"', '"record_group"', '"site"',
       '"identification_complete"', '"visit_date"', '"visit_datediff_s"',
       '"visit_datediff_c"', '"visit_date_mri"',
       ...
       '"study_retraction"', '"study_interruption_comm"',
       '"study_interruption_complete"', 'Unnamed: 650', 'Unnamed: 651',
       'Unnamed: 652', 'Unnamed: 653', 'Unnamed: 654', 'Unnamed: 655',
       'Unnamed: 656'],
      dtype='object', length=657)
0    "GC001"
1    "GC001"
2    "GC002"
3    "GC002"
4    "GC003"
Name: "record_id", dtype: object
  sub_id  session  risperidon_score bacs_total  bacs_att  panss_positive  \
0  GC001        1               0.0         38      68.0             7.0   
1  GC001        2               0.0         40      61.0             7.0   
2  GC002        1               0.0         55      63.0             7.0   
3  GC002        2               0.0         65      76.0             7.0   
4  GC003        1               0.0         66    

# HCP-EP

In [1]:
import os
import pandas as pd
from functools import reduce

# Directory containing the .xlsx files
directory = '/Volumes/Genf/HCP_EP/Metadata/Metadata_xlsx'

# List to store dataframes
dfs = []

# Set to track unmatched subject_ids across all files
unmatched_subjects = set()

# Dictionary to store missing subject IDs for each file (relative to baseline)
missing_subjects_per_file = {}

# -------------------------------------------------------------------
# 1) Build ordered file list with 'medication_information.xlsx' first
# -------------------------------------------------------------------
all_files = [
    f for f in os.listdir(directory)
    if f.endswith('.xlsx')
    and not f.startswith('._')
    and not f.startswith('~$')
]

baseline_filename = 'medication_information.xlsx'
if baseline_filename not in all_files:
    raise FileNotFoundError(f"'{baseline_filename}' not found in {directory}")

# Put baseline file first
all_files.remove(baseline_filename)
ordered_files = [baseline_filename] + sorted(all_files)

print("File processing order:")
for f in ordered_files:
    print("  -", f)

# -------------------------------------------------------------------
# 2) Loop through files in the desired order
# -------------------------------------------------------------------
baseline_subjects = None  # will be set from medication_information.xlsx

for idx, filename in enumerate(ordered_files):
    file_path = os.path.join(directory, filename)
    print(f"\nProcessing file: {filename}")

    # Read the Excel file into a DataFrame with the 'openpyxl' engine
    df = pd.read_excel(file_path, engine='openpyxl')

    if filename == "Motor_scores.xlsx":
        if "version_form" not in df.columns:
            print(f"Warning: 'version_form' column not found in {filename}, skipping this file.")
            continue

        target = "NIH Toolbox Grip Strength Test Age 3+ v2.0"
        # robust string handling
        df["version_form"] = df["version_form"].astype(str).str.strip()
        before = len(df)
        df = df[df["version_form"] == target].copy()
        after = len(df)
        print(f"Filtered {filename} on version_form == '{target}': {before} → {after} rows")

    # Check if 'src_subject_id' column exists
    if 'src_subject_id' not in df.columns:
        print(f"Warning: 'src_subject_id' column not found in {filename}, skipping.")
        continue

    # Standardize 'src_subject_id'
    df['src_subject_id'] = df['src_subject_id'].astype(str).str.strip()

    if idx == 0:
        # This is medication_information.xlsx → define baseline subjects
        baseline_subjects = set(df['src_subject_id'].unique())
        print(f"Baseline subjects from {filename}: {len(baseline_subjects)}")
    else:
        # For all other files: track which baseline subjects are missing in this file
        file_subjects = set(df['src_subject_id'].unique())
        missing_here = sorted(baseline_subjects - file_subjects)  # subject_ids in baseline but NOT in this file

        missing_subjects_per_file[filename] = missing_here
        unmatched_subjects.update(missing_here)

        print(f"Subjects in {filename}: {len(file_subjects)}")
        print(f"Missing baseline subjects in {filename}: {len(missing_here)}")

    # Append DataFrame to list for later merge
    dfs.append(df)

# -------------------------------------------------------------------
# 3) Merge all DataFrames on 'src_subject_id' without duplicates
#    (keep the first occurrence of each column)
# -------------------------------------------------------------------
if dfs:
    # Start from the first (baseline) dataframe
    result = dfs[0].copy()

    for df in dfs[1:]:
        # Keep only columns that are not already in result (except src_subject_id)
        new_cols = [
            c for c in df.columns
            if c == 'src_subject_id' or c not in result.columns
        ]

        df_reduced = df[new_cols]

        # Outer merge on src_subject_id
        result = pd.merge(result, df_reduced, on='src_subject_id', how='outer')

    print("\nFinal merged dataframe shape:", result.shape)
else:
    print("No matching Excel files found or processed.")

# Optional: save the updated dataframe to a new CSV file
# Create a new DataFrame by selecting the necessary columns and creating new ones
new_df = pd.DataFrame()

# Add new columns based on existing ones and apply the necessary transformations
new_df['CAINS_MAP'] = result['cains_map_ssum']
new_df['fluid_cognition_score'] = result['nih_fluidcogcomp_unadjusted']
new_df['crystal_cognition_score'] = result['nih_crycogcomp_unadjusted']
new_df['total_cognition_score'] = result['nih_totalcogcomp_unadjusted']
new_df['lifetime_antipsychotic_drug'] = result['apd_exp_months']
new_df['CPZ_score'] = result['apd_date_equiv']
new_df['money_delay_auc_200'] = result['auc_200']
new_df['money_delay_auc_40000'] = result['auc_40000']
new_df['group'] = result['phenotype_description']
new_df['pattern_processing_speed_NIH'] = result['nih_patterncomp_raw']
new_df['grip_strength_age_unadjusted'] = result['grip_standardsc_dom']

# Calculate sums for PANSS scores (positive, negative, general)
new_df['panss_positive'] = result[['pos_p1', 'pos_p2', 'pos_p3', 'pos_p4', 'pos_p5', 'pos_p6', 'pos_p7']].sum(axis=1)
new_df['panss_negative'] = result[['neg_n1', 'neg_n2', 'neg_n3', 'neg_n4', 'neg_n5', 'neg_n6', 'neg_n7']].sum(axis=1)
new_df['panss_negative_EPA'] = result[['neg_n1', 'neg_n2', 'neg_n3', 'neg_n4','neg_n6']].sum(axis=1)
new_df['panss_apathy_Indrit'] = result[['neg_n2', 'neg_n4']].sum(axis=1)
new_df['panss_general'] = result[['gps_g1', 'gps_g2', 'gps_g3', 'gps_g4', 'gps_g5', 'gps_g6', 'gps_g7']].sum(axis=1)

# Add the 'sub_id' column as a copy of 'src_subject_id'
new_df['sub_id'] = result['src_subject_id']

# Convert 'interview_age' to numeric (if it's not already), and calculate 'age_years' by dividing 'interview_age' by 12
new_df['interview_age'] = pd.to_numeric(result['interview_age'], errors='coerce')  # Convert to numeric, invalid parsing becomes NaN
new_df['age_years'] = new_df['interview_age'] / 12

# Convert 'sex' to lowercase
new_df['sex'] = result['sex'].str.lower()
new_df.to_excel('/Volumes/Genf/HCP_EP/Metadata/Metadata_xlsx/final_metadata.xlsx', index=False)
print(new_df['panss_apathy_Indrit'])

File processing order:
  - medication_information.xlsx
  - Motor_scores.xlsx
  - Processing_speed.xlsx
  - cains_score.xlsx
  - cognitive_data.xlsx
  - final_metadata.xlsx
  - metadata_final.xlsx
  - money_delay_task.xlsx
  - panss_score.xlsx
  - patient_or_control.xlsx
  - sociodemographic_data.xlsx

Processing file: medication_information.xlsx
Baseline subjects from medication_information.xlsx: 252

Processing file: Motor_scores.xlsx
Filtered Motor_scores.xlsx on version_form == 'NIH Toolbox Grip Strength Test Age 3+ v2.0': 469 → 233 rows
Subjects in Motor_scores.xlsx: 233
Missing baseline subjects in Motor_scores.xlsx: 19

Processing file: Processing_speed.xlsx
Subjects in Processing_speed.xlsx: 234
Missing baseline subjects in Processing_speed.xlsx: 18

Processing file: cains_score.xlsx
Subjects in cains_score.xlsx: 178
Missing baseline subjects in cains_score.xlsx: 74

Processing file: cognitive_data.xlsx
Subjects in cognitive_data.xlsx: 235
Missing baseline subjects in cognitive_

In [ ]:
import pandas as pd

# Assuming `result` is your existing DataFrame

# Create a new DataFrame by selecting the necessary columns and creating new ones
new_df = pd.DataFrame()

# Add new columns based on existing ones and apply the necessary transformations
new_df['CAINS_MAP'] = result['cains_map_ssum']
new_df['fluid_cognition_score'] = result['nih_fluidcogcomp_unadjusted']
new_df['crystal_cognition_score'] = result['nih_crycogcomp_unadjusted']
new_df['total_cognition_score'] = result['nih_totalcogcomp_unadjusted']
new_df['lifetime_antipsychotic_drug'] = result['apd_exp_months']
new_df['CPZ_score'] = result['apd_date_equiv']
new_df['money_delay_auc_200'] = result['auc_200']
new_df['money_delay_auc_40000'] = result['auc_40000']
new_df['group'] = result['phenotype']

# Calculate sums for PANSS scores (positive, negative, general)
new_df['panss_positive'] = result[['pos_p1', 'pos_p2', 'pos_p3', 'pos_p4', 'pos_p5', 'pos_p6', 'pos_p7']].sum(axis=1)
new_df['panss_negative'] = result[['neg_n1', 'neg_n2', 'neg_n3', 'neg_n4', 'neg_n5', 'neg_n6', 'neg_n7']].sum(axis=1)
new_df['panss_general'] = result[['gps_g1', 'gps_g2', 'gps_g3', 'gps_g4', 'gps_g5', 'gps_g6', 'gps_g7']].sum(axis=1)

# Add the 'sub_id' column as a copy of 'src_subject_id'
new_df['sub_id'] = result['src_subject_id']

new_df['age_years'] = pd.to_numeric(result['interview_age'], errors='coerce')/ 12  # Convert to numeric, invalid parsing becomes NaN

# Convert 'sex' to lowercase
new_df['sex'] = result['sex'].str.lower()

# Save the new dataframe as 'metadata_final.xlsx' in the specified folder
output_file_path = '/Volumes/Genf/HCP_EP/Metadata/Metadata_xlsx/metadata_final.xlsx'
new_df.to_excel(output_file_path, index=False)  # Save without the index column